# MCP client inside an agent

The pattern: the agent gets its tool list from `client.list_tools()` and executes them via `client.call_tool(name, args)`. The framework (LangGraph, OpenAI Agents SDK, ReAct loop) doesn't care that the tools live on an MCP server — they look like any other callable.

## Real LangGraph + langchain-mcp-adapters

```python
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

client = MultiServerMCPClient({'corpus': {'command': 'python', 'args': ['server.py'], 'transport': 'stdio'}})
tools = await client.get_tools()
agent = create_react_agent('openai:gpt-4o-mini', tools)
result = await agent.ainvoke({'messages': [{'role': 'user', 'content': 'Summarise RA-MoE.'}]})
```

Below we drive the in-process server via the same Client/Tool shape.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / '06-mcp'))
os.environ.setdefault('LLM_CACHE_ONLY', '1')

from mcp_core import build_corpus_server, Client, InProcessTransport, mcp_agent_solve
from shared.loaders import load_golden_qa

qa = {q.id: q for q in load_golden_qa()}
server = build_corpus_server(); client = Client(transport=InProcessTransport(server)); client.initialize()
out = mcp_agent_solve(client, qa['q01'].question)
for step in out['trace']:
    print(step['role'], '|', step.get('name') or '', '|', step.get('result_summary') or step.get('content', '')[:80])

Compare with `03-agentic-frameworks/00-react-from-scratch/eval-snapshot.json` — the metric shape is the same, so the MCP transport snapshot is directly comparable to the native ReAct snapshot. That's the point of MCP: identical agent semantics, swappable transport.